# Minimal Unsloth GRPO on Colab with a remote OpenEnv Space

This notebook is intentionally similar to the 2048 notebook pattern:
- training runs locally inside Colab
- the environment is accessed remotely through a Hugging Face Space
- the reward function is defined in notebook code by replaying actions against that remote env
- prompt / action / conclusion formatting mirrors the repo logic without importing the repo training script

Default remote env: `Ev3Dev/hackathon`

**Runtime**: Enable a GPU in Colab: Runtime -> Change runtime type -> GPU.

In [ ]:
# 1. Clone the repo for lightweight client / model definitions only
REPO_URL = "https://github.com/mhtruong1031/OpenENV-Hackathon.git"  # or your fork
REPO_DIR = "OpenENV-Hackathon"

!git clone --depth 1 {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}

In [ ]:
# 2. Install only the runtime pieces needed for notebook-side training
!pip install -q unsloth unsloth_zoo --no-deps
!pip install -q "openenv-core[core]>=0.2.0" "pydantic>=2" "numpy>=1.24.0" "scipy>=1.10.0" "datasets>=4.6.1" "accelerate>=1.13.0" "peft>=0.15.0" "bitsandbytes>=0.45.0" "matplotlib>=3.8.0"
!pip install -q "transformers>=4.57.0" "trl>=0.29.0" "torchvision>=0.20.0"

In [ ]:
# 3. Import repo reward helpers, but keep the environment remote
import inspect
import json
import random
import sys
from pathlib import Path
from typing import Any, Dict, List

# Unsloth must be imported before trl / transformers / peft.
import unsloth  # noqa: F401
import torch
from unsloth import FastLanguageModel, PatchFastRL

sys.path.insert(0, str(Path.cwd()))

from client import BioExperimentEnv
from models import ActionType, ExperimentAction
from training_script import (
    INVALID_ACTION_PENALTY,
    ENVIRONMENT_ERROR_PENALTY,
    OpenEnvReward,
    build_training_prompt,
    build_experiment_action,
    decode_history_actions,
    pick_action,
    save_training_plots,
)

MAX_COMPLETION_TOKENS = 160
LORA_TARGET_MODULES = [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
    "gate_proj",
    "up_proj",
    "down_proj",
]


def hf_space_repo_to_base_url(repo_id: str) -> str:
    owner, space_name = repo_id.split("/", 1)
    return f"https://{owner.lower().replace('_', '-')}-{space_name.lower().replace('_', '-')}.hf.space"


def build_remote_prompt_examples(
    base_url: str,
    dataset_episodes: int,
    rollout_steps: int,
    seed: int,
) -> List[Dict[str, str]]:
    rng = random.Random(seed)
    examples: List[Dict[str, str]] = []

    for _ in range(dataset_episodes):
        with BioExperimentEnv(base_url=base_url) as env:
            result = env.reset()
            obs = result.observation
            history_actions: List[ExperimentAction] = []

            for step_idx in range(rollout_steps):
                if obs.done:
                    break

                next_action = build_experiment_action(
                    action_type=pick_action(
                        "heuristic",
                        step_idx,
                        [action.action_type for action in history_actions],
                    ),
                    discovered_markers=obs.discovered_markers,
                    candidate_mechanisms=obs.candidate_mechanisms,
                    conditions=obs.task.conditions,
                )
                examples.append(
                    {
                        "prompt": build_training_prompt(obs),
                        "history_actions": json.dumps(
                            [action.model_dump() for action in history_actions]
                        ),
                        "reference_action": json.dumps(next_action.model_dump()),
                        "problem_statement": obs.task.problem_statement,
                        "episode_tag": f"remote-{rng.randrange(10**9):09d}",
                    }
                )

                history_actions.append(next_action)
                result = env.step(next_action)
                obs = result.observation
                if result.done:
                    break

    return examples


def build_grpo_config(**overrides: Any):
    from trl import GRPOConfig

    supported = set(inspect.signature(GRPOConfig.__init__).parameters)
    config_kwargs = {
        "output_dir": overrides["output_dir"],
        "learning_rate": overrides["learning_rate"],
        "per_device_train_batch_size": overrides["per_device_train_batch_size"],
        "gradient_accumulation_steps": overrides["gradient_accumulation_steps"],
        "num_generations": overrides["num_generations"],
        "max_completion_length": overrides["max_completion_length"],
        "num_train_epochs": overrides["num_train_epochs"],
        "logging_steps": overrides["logging_steps"],
        "save_steps": overrides["save_steps"],
        "bf16": overrides["bf16"],
        "fp16": overrides["fp16"],
        "report_to": "none",
        "remove_unused_columns": False,
    }
    # Keep prompt truncation enabled. Leaving this as None can trigger
    # an Unsloth rotary-cache shape mismatch on long GRPO prompts.
    if "max_prompt_length" in supported:
        config_kwargs["max_prompt_length"] = overrides["max_prompt_length"]
    if (
        "max_length" in supported
        and "max_prompt_length" not in supported
        and "max_completion_length" not in supported
    ):
        config_kwargs["max_length"] = (
            overrides["max_prompt_length"] + overrides["max_completion_length"]
        )
    return GRPOConfig(**{k: v for k, v in config_kwargs.items() if k in supported})


print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
Path("artifacts").mkdir(exist_ok=True)

In [ ]:
# 4. Config + collect prompt states from the remote Space
SPACE_REPO_ID = "Ev3Dev/hackathon"
SPACE_BASE_URL = hf_space_repo_to_base_url(SPACE_REPO_ID)
# If your Space has a custom domain, replace SPACE_BASE_URL manually.

MODEL_ID = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit"
OUTPUT_DIR = "artifacts/grpo-unsloth-llama32-3b-remote-space"

DATASET_EPISODES = 8
ROLLOUT_STEPS = 6
NUM_GENERATIONS = 2
# Keep this modest for Unsloth GRPO stability on Colab.
MAX_PROMPT_LENGTH = 768
MAX_SEQ_LENGTH = 2048
PER_DEVICE_TRAIN_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 4
LEARNING_RATE = 5e-6
NUM_TRAIN_EPOCHS = 1.0
LOGGING_STEPS = 1
SAVE_STEPS = 25
SEED = 42
LORA_R = 16
LORA_ALPHA = 16
LORA_DROPOUT = 0.0

examples = build_remote_prompt_examples(
    base_url=SPACE_BASE_URL,
    dataset_episodes=DATASET_EPISODES,
    rollout_steps=ROLLOUT_STEPS,
    seed=SEED,
)

reward_fn = OpenEnvReward(
    reward_backend="remote",
    base_url=SPACE_BASE_URL,
    invalid_action_penalty=INVALID_ACTION_PENALTY,
    environment_error_penalty=ENVIRONMENT_ERROR_PENALTY,
)

print("Remote env:", SPACE_BASE_URL)
print("Prompt states:", len(examples))
print("Sample prompt preview:\n")
print(examples[0]["prompt"][:2000])

In [ ]:
# 5. Local GRPO training in Colab, remote env for rewards
from datasets import Dataset
from trl import GRPOTrainer

PatchFastRL("GRPO", FastLanguageModel)
train_dataset = Dataset.from_list(examples)

bf16 = bool(getattr(torch.cuda, "is_bf16_supported", lambda: False)()) if torch.cuda.is_available() else False
runtime_dtype = torch.bfloat16 if bf16 else (torch.float16 if torch.cuda.is_available() else torch.float32)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=runtime_dtype,
    load_in_4bit=True,
)
if tokenizer.pad_token is None and tokenizer.eos_token is not None:
    tokenizer.pad_token = tokenizer.eos_token

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=LORA_TARGET_MODULES,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    use_gradient_checkpointing=True,
    random_state=SEED,
)

training_args = build_grpo_config(
    output_dir=OUTPUT_DIR,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    num_generations=NUM_GENERATIONS,
    max_completion_length=MAX_COMPLETION_TOKENS,
    max_prompt_length=MAX_PROMPT_LENGTH,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    logging_steps=LOGGING_STEPS,
    save_steps=SAVE_STEPS,
    bf16=bf16,
    fp16=torch.cuda.is_available() and not bf16,
)

trainer = GRPOTrainer(
    model=model,
    reward_funcs=[reward_fn],
    args=training_args,
    train_dataset=train_dataset,
    processing_class=tokenizer,
)

for attr in ("image_token_id", "vision_start_token_id", "vision_end_token_id"):
    if not hasattr(trainer, attr):
        setattr(trainer, attr, None)

trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
plot_paths = save_training_plots(trainer.state.log_history, OUTPUT_DIR)

result = {
    "trainer": trainer,
    "plot_paths": plot_paths,
    "output_dir": OUTPUT_DIR,
}
print("Saved to:", OUTPUT_DIR)
print("Plots:", plot_paths)

In [ ]:
# 6. (Optional) Show curves and sanity-check the repo reward wrapper
from IPython.display import Image, display

sample_reward = reward_fn(
    completions=[[{"role": "assistant", "content": examples[0]["reference_action"]}]],
    history_actions=[examples[0]["history_actions"]],
)[0]
print("Sample reward for reference action:", sample_reward)

for name, path in (result.get("plot_paths") or {}).items():
    print(name, path)
    display(Image(filename=path))